# Carga da série histórica nacional (NEX-GDDP-CMIP6)

Parte 1 — setup: imports e conexões (S3 + Postgres).


In [6]:
import os 
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import xarray as xr
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from botocore.config import Config
import sys
sys.path.append("../scripts") # achar o .py

from recortar_brasil import recortar_brasil

In [17]:
load_dotenv()

BUCKET = "nex-gddp-cmip6"
BASE_PREFIX = "NEX-GDDP-CMIP6/"

client = boto3.client(
    "s3",
    config=Config(signature_version=UNSIGNED, connect_timeout=10, read_timeout=30, retries={"max_attempts": 2})
)
usuario = os.environ["DB_USER"]
senha = os.environ["DB_PASSWORD"]
host = os.environ.get("DB_HOST", "localhost")
porta = os.environ.get("DB_PORT", "5432")
nome_banco = os.environ["DB_NAME"]
engine = create_engine(f"postgresql+psycopg2://{usuario}:{senha}@{host}:{porta}/{nome_banco}")

# Corrige o caminho garantindo que a pasta exista
os.makedirs("../data/raw", exist_ok=True)
destino_tmp = "../data/raw/_tmp_ano.nc"

In [18]:
# teste de conexão
engine.connect()
print("conectado")

conectado


#### Funções de baixar ano e agregar para série nacional


In [19]:

def baixar_ano(client, model, scenario, variable, ano, destino_tmp):
  for sufixo in ["_v2.0", "_v1.1", ""]:
    print("ok")
    key = f"{BASE_PREFIX}{model}/{scenario}/r1i1p1f1/{variable}/{variable}_day_{model}_{scenario}_r1i1p1f1_gn_{ano}{sufixo}.nc"
    try:
      client.download_file(BUCKET, key, destino_tmp)
      return key
    except Exception:
      continue
  raise FileNotFoundError(f"Nenhuma verão encontrada para {variable}/{ano}")

In [20]:
def agregar_nacional(destino_tmp, variable):
    ds = xr.open_dataset(destino_tmp)
    ds_brasil = recortar_brasil(ds)
    da = ds_brasil[variable]
    media_diaria = da.mean(dim=["lat", "lon"], skipna=True)
    df = media_diaria.to_dataframe().reset_index()
    df = df.rename(columns={variable: "valor"})
    df["variavel"] = variable
    return df[["time", "variavel", "valor"]]

### Loop de teste que baixa, agrega e carrega no banco


In [21]:
model = "ACCESS-CM2"
scenario = "historical"
variable = "pr"
ano_incio = 1950
ano_fim = 1954
tabela = "clima_diario_nacional"

for ano in range(ano_incio, ano_fim + 1):
  print(f"Processando {variable} {ano}...")
  try:
    baixar_ano(client, model, scenario, variable, ano, destino_tmp)
    df = agregar_nacional(destino_tmp, variable)
    df["modelo"] = model
    df.to_sql(tabela, engine, if_exists="append", index=False)
  except Exception as e:
    print(f" Erro: {ano} - {e}")
  finally:
    if os.path.exists(destino_tmp):
      os.remove(destino_tmp)

print("Download concluido")

Processando pr 1950...
ok
Processando pr 1951...
ok
Processando pr 1952...
ok
Processando pr 1953...
ok
Processando pr 1954...
ok
Download concluido


confirmando se funcionou, ja que o codigo não retornou as linhas por ano


In [22]:
pd.read_sql("SELECT COUNT(*) FROM clima_diario_nacional", engine)

,count
0,1826


In [23]:
pd.read_sql("SELECT * FROM clima_diario_nacional ORDER BY time LIMIT 5", engine)

,time,variavel,valor,modelo
0,1950-01-01 12:00:00,pr,0.000077,ACCESS-CM2
1,1950-01-02 12:00:00,pr,0.000079,ACCESS-CM2
2,1950-01-03 12:00:00,pr,0.000077,ACCESS-CM2
3,1950-01-04 12:00:00,pr,0.000083,ACCESS-CM2
4,1950-01-05 12:00:00,pr,0.000083,ACCESS-CM2
